# VisiPruner trên SmolVLM · Phần 2/2

Chạy cấu hình **13–23**, sau đó chấm điểm và vẽ hình cho cả 23 cấu hình. Dùng cùng loại GPU và cấu hình sinh như phần 1. Trong phiên mới, gắn output phần 1 làm Kaggle Input. Khi có nhiều output phần 1, đặt `PART1_OUTPUT` thành đường dẫn chứa `manifest.json`.

In [ ]:
%%writefile /kaggle/working/visipruner-ablation-part2.env
REPOSITORY=https://github.com/huyvanzzz/Log_Metrics.git
SOURCE_BRANCH=smolvlm-attention-nguyet
SOURCE_DIR=/kaggle/working/log-metrics-visipruner
DATA_DIR=/kaggle/working/drivelm-nuscenes-v1-0
export PART1_OUTPUT=''
OUTPUT_DIR=/kaggle/working/visipruner_ablation_outputs
BATCH_SIZE=8
MAX_NEW_TOKENS=128


## Cài đặt

Cập nhật mã nguồn và cài các thư viện dùng để chạy và chấm điểm.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
if [ -d "$SOURCE_DIR/.git" ]; then
  test "$(git -C "$SOURCE_DIR" branch --show-current)" = "$SOURCE_BRANCH"
  test -z "$(git -C "$SOURCE_DIR" status --porcelain --untracked-files=no)"
  git -C "$SOURCE_DIR" pull --ff-only origin "$SOURCE_BRANCH"
else
  git clone --depth 1 --single-branch --branch "$SOURCE_BRANCH" "$REPOSITORY" "$SOURCE_DIR"
fi
python -m pip install -q -r "$SOURCE_DIR/smolvlm_attention/requirements.txt" pycocoevalcap==1.2
if ! command -v java >/dev/null; then
  apt-get update -qq
  apt-get install -y -qq default-jre-headless
fi


## Khôi phục kết quả phần 1

Kiểm tra đủ kết quả, thứ hạng baseline và cache đặc trưng ảnh. Có thể dùng trực tiếp kết quả nếu chạy trong cùng phiên. Trong phiên mới, bước này sao chép dữ liệu cần thiết sang thư mục làm việc.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -m smolvlm_attention.visipruner_ablation handoff --output "$OUTPUT_DIR"


## 13/23 · Layer 18–24 · Mask top 10%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant middle-18-24-top10 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 14/23 · Layer 18–24 · Mask bottom 10%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant middle-18-24-bottom10 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 15/23 · Layer 18–24 · Chỉ giữ top 5%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant middle-18-24-keep5 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 16/23 · Layer 18–24 · Mask top 10% và bảo toàn visual mass

Mask token đã chọn, sau đó phân bổ lại trọng số lên các token ảnh còn lại. Giữ nguyên trọng số lên text.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant middle-18-24-top10-mass --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 17/23 · Layer 18–24 · Mask bottom 10% và bảo toàn visual mass

Mask token đã chọn, sau đó phân bổ lại trọng số lên các token ảnh còn lại. Giữ nguyên trọng số lên text.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant middle-18-24-bottom10-mass --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 18/23 · Layer 20 · Skip token ảnh

Bỏ qua xử lý visual tokens tại đúng layer này. Các visual tokens quay lại ở layer kế tiếp.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant deep-skip-20 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 19/23 · Layer 20 · Discard token ảnh

Loại visual tokens khỏi chuỗi và KV cache từ layer này đến hết.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant deep-discard-20 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 20/23 · Layer 23 · Skip token ảnh

Bỏ qua xử lý visual tokens tại đúng layer này. Các visual tokens quay lại ở layer kế tiếp.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant deep-skip-23 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 21/23 · Layer 23 · Discard token ảnh

Loại visual tokens khỏi chuỗi và KV cache từ layer này đến hết.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant deep-discard-23 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 22/23 · Layer 26 · Skip token ảnh

Bỏ qua xử lý visual tokens tại đúng layer này. Các visual tokens quay lại ở layer kế tiếp.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant deep-skip-26 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 23/23 · Layer 26 · Discard token ảnh

Loại visual tokens khỏi chuỗi và KV cache từ layer này đến hết.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant deep-discard-26 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Chấm điểm và vẽ hình

Xuất bảng điểm và bốn hình PNG/PDF. Bước này chạy trên CPU.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation score --output "$OUTPUT_DIR"


In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation-part2.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation plot --output "$OUTPUT_DIR"
